In [1]:
import os
import pickle
from astropy.io import fits
from astropy.cosmology import FlatLambdaCDM
from astropy.units import Quantity
from astropy.table import Table, vstack
from tqdm import tqdm

import slsim.Sources as sources
from utils import extract_non_lens_properties, make_multiband_images_and_rgb_image, plot_montage

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

# Define and create output directories
img_dir = "../data/nonlenses/images/"
table_dir = "../data/nonlenses/"
os.makedirs(img_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

In [2]:
# ==========================================
# Cell 2: Loader Helper and Load Non-Lenses
# ==========================================
def load_objects(filepath):
    if os.path.exists(filepath):
        print(f"Loading {filepath}...")
        with open(filepath, "rb") as f:
            return pickle.load(f)
    else:
        print(f"Warning: {filepath} not found. Skipping...")
        return []

print("Loading all non-lens categories...")

# Categories WITH rendered central object light
gal_gal_agn = load_objects("../saved_objects/nonlenses_gal_gal_agn_10000.pkl")
qso_gal = load_objects("../saved_objects/nonlenses_qso_gal_10000.pkl")
# cusp_mimic = load_objects("../saved_objects/nonlenses_cusp_mimic_10000.pkl")

all_with_central = gal_gal_agn + qso_gal #+ cusp_mimic

# Categories WITHOUT rendered central object light
qso_qso = load_objects("../saved_objects/nonlenses_qso_qso_10000.pkl")
qso_star = load_objects("../saved_objects/nonlenses_qso_star_10000.pkl")
star_star = load_objects("../saved_objects/nonlenses_star_star_10000.pkl")

all_without_central = qso_qso + qso_star + star_star

all_nonlenses = all_with_central + all_without_central
print(f"\nLoaded {len(all_nonlenses)} total non-lenses.")

Loading all non-lens categories...
Loading ../saved_objects/nonlenses_gal_gal_agn_10000.pkl...
Loading ../saved_objects/nonlenses_qso_gal_10000.pkl...
Loading ../saved_objects/nonlenses_qso_qso_10000.pkl...
Loading ../saved_objects/nonlenses_qso_star_10000.pkl...
Loading ../saved_objects/nonlenses_star_star_10000.pkl...

Loaded 50000 total non-lenses.


In [12]:
# ==========================================
# Cell 3: Extract Properties and Save Table
# ==========================================
print("Extracting physical properties into catalog table...")

# Extract maintaining distinct IDs using starting_index
table_with_central = extract_non_lens_properties(
    all_with_central, all_bands=['g', 'r', 'i', 'z', 'y'], 
    has_central_object=True, starting_index=0
)

table_without_central = extract_non_lens_properties(
    all_without_central, all_bands=['g', 'r', 'i', 'z', 'y'], 
    has_central_object=False, starting_index=len(all_with_central)
)

table_all_nonlenses = vstack([table_with_central, table_without_central])

table_path = os.path.join(table_dir, f"table_all_nonlenses_{len(table_all_nonlenses)}.fits")
table_all_nonlenses.write(table_path, format="fits", overwrite=True)
print(f"Saved catalog to {table_path}")

Extracting physical properties into catalog table...
Saved catalog to ../data/nonlenses/table_all_nonlenses_50000.fits


In [13]:
table_all_nonlenses

Object ID,RA,Dec,z_central,mag_lens_g,mag_lens_r,mag_lens_i,mag_lens_z,mag_lens_y
str12,float64,float64,float64,float64,float64,float64,float64,float64
D1_N00000000,118.27319900405493,23.190738435810424,0.6551609940261384,28.065419420372493,26.804820496089093,25.718149849167276,25.31441991109898,25.01837787740702
D1_N00000001,25.682450076045136,12.751781379440692,0.9474052864418853,24.814982684291362,22.75354888284325,21.666548042058654,20.739344876272117,20.34281328923161
D1_N00000002,257.8621851221455,-41.142683264936,0.9029296341342866,22.941304001330494,22.69174794082236,22.105073585475253,21.56956209898108,21.289983582816667
D1_N00000003,49.94956108569791,-17.494334428347653,0.5485393851896125,23.752051861020274,22.53185062138544,21.63059031916878,21.241949289687387,20.981590063896828
D1_N00000004,5.965863857892688,-72.80673668841811,0.4805397306550294,27.7581446050838,26.339958078764866,25.55096583471495,25.185871984644784,24.976528100231683
D1_N00000005,99.95764139117385,2.98710954256864,0.585082835869243,27.94429069104571,26.683896185110804,25.665649019255838,25.2850609806678,25.023637357380807
D1_N00000006,56.438723829072195,-59.283299178596444,0.5489621103817292,23.469865206662533,22.25005744080869,21.357975086626496,20.985384905259394,20.73816762106094
D1_N00000007,229.22895258001162,19.300648965043504,2.3015901687726026,25.701244092723684,25.556627475685517,25.424793672226357,25.24628364176072,24.703929682922038
D1_N00000008,256.5739234511634,82.49786046567138,0.4485714384231576,22.17018874001743,20.729842003797764,20.059254829527802,19.725193926018726,19.51017409830741


In [14]:
# ==========================================
# Cell 4: Setup Field Galaxy Population and Inject
# ==========================================
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
sky_area_galaxy = Quantity(2, "deg2") 

print("\nLoading SkyPy catalog for field galaxies...")
all_galaxy_catalog = Table.read(f'../catalogs/skypy_all_galaxies_{sky_area_galaxy.value}deg2.fits', format='fits')

field_galaxy_pop = sources.Galaxies(
    galaxy_list=all_galaxy_catalog, 
    kwargs_cut={"band": "i", "band_max": 26, "z_min": 0.01, "z_max": 5.0}, 
    cosmo=cosmo, 
    sky_area=sky_area_galaxy, 
    catalog_type="skypy"
)

print("Injecting field galaxies into non-lens cutouts (Area: 70 arcsec^2)...")
for obj in all_nonlenses:
    obj.add_field_galaxies(field_galaxies=field_galaxy_pop.draw_galaxies(area=Quantity(70, "arcsec2")))


Loading SkyPy catalog for field galaxies...
Injecting field galaxies into non-lens cutouts (Area: 70 arcsec^2)...


In [ ]:
# ==========================================
# Cell 5: Render and Save FITS Images
# ==========================================
print("\nRendering images (coadd_years=1) and saving to disk. This will take time...")

for obj, row in tqdm(zip(all_nonlenses, table_all_nonlenses), total=len(table_all_nonlenses), desc="Processing non-lenses"):
    obj_id = row["Object ID"]
    
    # Generate 1-year co-add images
    multiband_image, _ = make_multiband_images_and_rgb_image(
        lens_class=obj,
        bands=['g', 'r', 'i', 'z', 'y'],
        num_pix=41,
        coadd_years=1, 
        add_noise=True,
        rgb_bands=['i', 'r', 'g'],
        rgb_stretch=0.5,
    )

    for band in ['g', 'r', 'i', 'z', 'y']:
        fits_path = os.path.join(img_dir, f"{obj_id}_{band}.fits")
        fits.writeto(fits_path, multiband_image[band], overwrite=True)

print("Non-Lens rendering and extraction complete!")